# ComfyUI Master Pipeline on Google Colab Pro
### 堅牢・高速・完全自動化された画像/動画生成パイプライン

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/houjiro/googlecolab-by-ai/blob/main/ComfyUI_Colab_Master.ipynb)

このノートブックは、Google Colab ProのリソースとGoogle Driveを連携させ、ComfyUIを用いた自律生成を行うための統合パイプラインです。

## 0. リポジトリの同期 (Colab初回実行用)
GitHubリポジトリ (`googlecolab-by-ai`) をColab環境内に同期し、最新のスクリプト群を作業ディレクトリに読み込みます。

In [ ]:
# リポジトリの取得とカレントディレクトリの設定
import os
if not os.path.exists("/content/googlecolab-by-ai"):
    !git clone https://github.com/houjiro/googlecolab-by-ai.git /content/googlecolab-by-ai
else:
    !git -C /content/googlecolab-by-ai pull
%cd /content/googlecolab-by-ai

## 1. 【フェーズ1】環境初期化・高速起動
- Google Driveマウント（指定ドライブフォルダへの自動接続）
- Miniconda (Python 3.10) 仮想環境の展開/構築
- PyTorch / xformers / ComfyUI のインストールとDriveキャッシュ（高速復元対応）

In [ ]:
# setup_comfyui.py の読み込みと実行
%run setup_comfyui.py

## 2. 【フェーズ1・2】ComfyUI サーバーの起動 (バックグラウンド)
- ローカルAPI (http://127.0.0.1:8188) として起動
- 出力画像は自動的に Google Drive の `outputs` フォルダに保存されます

In [ ]:
import subprocess
import time
import urllib.request
from setup_comfyui import DRIVE_MASTER_DIR

conda_python = "/content/miniconda/envs/comfy_env/bin/python"
comfy_dir = "/content/ComfyUI"
output_dir = str(DRIVE_MASTER_DIR / "outputs")

print(f"Starting ComfyUI server in the background (Output: {output_dir})...")
comfy_cmd = [
    conda_python,
    f"{comfy_dir}/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--output-directory", output_dir,
    "--preview-method", "auto",
    "--highvram"
]

with open("/content/comfyui.log", "w") as log_file:
    comfy_proc = subprocess.Popen(comfy_cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("Waiting for ComfyUI to become ready...")
server_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1)

if server_ready:
    print("\n[SUCCESS] ComfyUI server is LIVE at http://127.0.0.1:8188 !")
else:
    print("\n[WARN] Server check timed out. Last 30 lines of log:")
    !tail -n 30 /content/comfyui.log

## 3. 【フェーズ2・3】APIラッパー関数のロード
AIオペレーターが自律操作するための関数を定義します:
- `install_model(url_or_hf_id, target_dir)`: モデル/LoRA/VAEのDriveへの配置
- `install_custom_node(git_url)`: ComfyUIカスタムノードの導入
- `generate_media(prompt_json)`: ComfyUIローカルAPIへのジョブ投入と進捗監視
- `preview_image(image_path)`: ノートブック上への直接画像/動画プレビュー表示

In [ ]:
# pipeline_tools.py から関数をインポート
from pipeline_tools import (
    install_model,
    install_custom_node,
    generate_media,
    preview_image,
    create_default_txt2img_workflow,
    create_sdxl_txt2img_workflow,
    get_token,
    require_hf_token,
    DRIVE_MASTER_DIR
)
print(f"[OK] Pipeline tools loaded. Active Drive Master: {DRIVE_MASTER_DIR}")

## 4. モデルの導入 (SDXL Base 1.0 & VAE)
Stable Diffusion XL Base 1.0 (`sd_xl_base_1.0.safetensors` 約6.9GB) および SDXL VAE を Drive に高速並列ダウンロードします。

In [ ]:
# download_sdxl.py の実行
%run download_sdxl.py

## 5. 【フェーズ4】自律運用モード (画像生成の実行)
プロンプトを投入して生成を実行し、生成結果を直下にプレビューします。

In [ ]:
# SDXL によるテスト生成例
prompt = "A serene cinematic sunset over a calm ocean, golden hour reflections on gentle waves, highly detailed, 8k resolution, masterpiece"
negative = "ugly, deformed, blurry, low quality, artifacts, bad anatomy, text, watermark"

print("Generating SDXL workflow...")
workflow = create_sdxl_txt2img_workflow(
    ckpt_name="sd_xl_base_1.0.safetensors",
    positive_prompt=prompt,
    negative_prompt=negative,
    width=1024,
    height=1024,
    steps=30,
    cfg=7.0
)

output_images = generate_media(workflow)
for img_file in output_images:
    preview_image(img_file)